In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h_big = 5
w_big = 5

h_small = 3
w_small = 3

avg_len = 0.8

In [ ]:
temp_ipu, points, segment_edges, m, marker= periodic_unit_helper.get_boundary_aligned_dashline(w_small, w_big, h_small, h_big, avg_len)

In [ ]:
visualization.plot_line_segments(points, segment_edges)

In [ ]:
finalMarkers = np.where(np.array(marker) == 1)[0]

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
len(m.vertices())

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
fuse_boundary = False

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
# fd_validation.gradConvergencePlot(az_ipu)

In [ ]:
# fd_validation.hessConvergencePlot(az_ipu)

In [ ]:
az_ipu = inflation.InflatableMidSurfacePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(az_ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])



In [ ]:
xy_fixedVars = periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[:2]

In [ ]:
# Choose strategy for constraining rigid motion
# fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6
# fixedVars, hessianShift = [], 1e-6

In [ ]:
az_ipu.ipu.sheet.setUseTensionFieldEnergy(False)
az_ipu.ipu.sheet.setUseHessianProjectedEnergy(False)
# az_ipu.ipu.sheet.disableFusedRegionTensionFieldTheory(False)
az_ipu.ipu.sheet.pressure = 1

In [ ]:
# ipu.gradient()

In [ ]:
az_ipu.ipu.energy()

In [ ]:
az_ipu.ipu.sheet.pressure

In [ ]:
az_ipu.ipu.sheet.energy()

In [ ]:
benchmark.reset()

opts.niter = 1000
opts.gradTol = 1e-10
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])
optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr = optimizer.optimize()
benchmark.report()

In [ ]:
cr.success

In [ ]:
gamma = np.mean(az_ipu.ipu.get_x_flat().reshape(int(ipu.sheet.numVars() / 3), 3)[:, 2])

In [ ]:
gamma

In [ ]:
import periodic_simulation_setup

In [ ]:
az_ipu.energy()

In [ ]:
curr_vars = az_ipu.getVars()

In [ ]:
az_ipu.energy()

In [ ]:
az_ipu.setVars(curr_vars)

In [ ]:
periodic_unit_helper.get_center_fixedVars(az_ipu.ipu)[:2]

In [ ]:
benchmark.reset()
stiffness, sampled_alphas = periodic_simulation_setup.visualize_sampled_bending_stiffness(az_ipu, 100, optimizer, filename = "stiffness_shifted_dashline_{}_average.png".format(0), hessianShift = 1e-10, fixedVars = xy_fixedVars)
benchmark.report()

In [ ]:
az_ipu.ipu.get_kappa()

In [ ]:
importlib.reload(periodic_simulation_setup)

In [ ]:
bs_obj = periodic_simulation_setup.bending_stiffness_class(az_ipu, az_ipu.ipu.sheet, optimizer, viewer, hessianShift = 1e-10, fixedVars = [])

bs_obj.setVars(bs_obj.getVars())

fd_validation.secondDerivativeConvergencePlot(bs_obj, epsilons = np.logspace(-6, -1, 100))

In [ ]:
fd_validation.secondDerivativeConvergencePlot(periodic_simulation_setup.bending_test_class(az_ipu), epsilons = np.logspace(-5, -3, 100))

In [ ]:
fd_validation.gradConvergencePlot(periodic_simulation_setup.bending_total_gradient(az_ipu, az_ipu.ipu.sheet, optimizer, viewer), epsilons = np.logspace(-5, -3, 100))